# CelebA Setup

This notebook downloads CelebA from Kaggle into `local_datasets/celeba` and checks the dataset format used by your main notebook.


## Before Running

Place your Kaggle token at `C:/Users/farih/.kaggle/kaggle.json`, or set `KAGGLE_USERNAME` and `KAGGLE_KEY`.


In [1]:
import shutil

import subprocess

import sys

import zipfile

from pathlib import Path



project_root = Path.cwd()

dataset_root = project_root / 'local_datasets' / 'celeba'

download_root = project_root / 'local_datasets' / '_downloads' / 'celeba'

archive_path = download_root / 'celeba-dataset.zip'

inner_zip_path = download_root / 'img_align_celeba.zip'

image_dir = dataset_root / 'img_align_celeba'



dataset_root.mkdir(parents=True, exist_ok=True)

download_root.mkdir(parents=True, exist_ok=True)



print('Dataset root:', dataset_root)


Dataset root: d:\THESIS\RandomizedSmothingmanifold\local_datasets\celeba


In [5]:
import importlib.util

import os

from pathlib import Path



if importlib.util.find_spec('kaggle') is None:

    subprocess.run([sys.executable, '-m', 'pip', 'install', 'kaggle'], check=True)



if image_dir.exists() and any(image_dir.glob('*.jpg')):

    print('CelebA images already exist.')

else:

    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'

    has_env_auth = bool(os.environ.get('KAGGLE_USERNAME')) and bool(os.environ.get('KAGGLE_KEY'))

    if not kaggle_json.exists() and not has_env_auth:

        raise RuntimeError(

            'Kaggle auth not found. Put kaggle.json in C:/Users/farih/.kaggle/ or set KAGGLE_USERNAME and KAGGLE_KEY.'

        )



    cmd = [

        'kaggle', 'datasets', 'download',

        '-d', 'jessicali9530/celeba-dataset',

        '-p', str(download_root),

        '--force'

    ]



    result = subprocess.run(

        cmd,

        capture_output=True,

        text=True,

        encoding='utf-8',

        errors='replace'

    )

    if result.returncode != 0:

        print('Kaggle command failed.')

        if result.stdout:

            print('stdout:\n', result.stdout)

        if result.stderr:

            print('stderr:\n', result.stderr)

        raise RuntimeError('Kaggle download failed. Check auth/token and dataset access permissions.')



    downloaded_files = sorted(download_root.glob('*.zip'))

    if not downloaded_files:

        raise FileNotFoundError('No zip file was downloaded.')



    latest_zip = max(downloaded_files, key=lambda path: path.stat().st_mtime)

    if latest_zip != archive_path:

        shutil.move(str(latest_zip), str(archive_path))



print('Archive ready:', archive_path)


Archive ready: d:\THESIS\RandomizedSmothingmanifold\local_datasets\_downloads\celeba\celeba-dataset.zip


In [7]:
import zipfile

with zipfile.ZipFile(archive_path, 'r') as z:
    top_level = set()
    for name in z.namelist():
        parts = name.split('/')
        if parts[0]:
            top_level.add(parts[0])
    
    print('Top-level items in archive:')
    for item in sorted(top_level):
        print(' -', item)
    
    print('\nFirst 10 file paths:')
    for name in z.namelist()[:10]:
        print(' -', name)


Top-level items in archive:
 - img_align_celeba
 - list_attr_celeba.csv
 - list_bbox_celeba.csv
 - list_eval_partition.csv
 - list_landmarks_align_celeba.csv

First 10 file paths:
 - img_align_celeba/img_align_celeba/000001.jpg
 - img_align_celeba/img_align_celeba/000002.jpg
 - img_align_celeba/img_align_celeba/000003.jpg
 - img_align_celeba/img_align_celeba/000004.jpg
 - img_align_celeba/img_align_celeba/000005.jpg
 - img_align_celeba/img_align_celeba/000006.jpg
 - img_align_celeba/img_align_celeba/000007.jpg
 - img_align_celeba/img_align_celeba/000008.jpg
 - img_align_celeba/img_align_celeba/000009.jpg
 - img_align_celeba/img_align_celeba/000010.jpg


In [9]:
import shutil

if image_dir.exists() and any(image_dir.glob('*.jpg')):
    print('Image folder already extracted.')
else:
    with zipfile.ZipFile(archive_path, 'r') as z:
        z.extractall(path=dataset_root)
    
    extracted_nested = dataset_root / 'img_align_celeba' / 'img_align_celeba'
    if extracted_nested.exists():
        temp_dir = dataset_root / 'img_align_celeba_temp'
        shutil.move(str(extracted_nested), str(temp_dir))
        shutil.rmtree(str(dataset_root / 'img_align_celeba'))
        shutil.move(str(temp_dir), str(image_dir))

print('Images ready:', image_dir)
print('Found', len(list(image_dir.glob('*.jpg'))), 'jpg files')


Images ready: d:\THESIS\RandomizedSmothingmanifold\local_datasets\celeba\img_align_celeba
Found 202599 jpg files


In [10]:
import torchvision

from torchvision import transforms



transform = transforms.Compose([

    transforms.Resize(64),

    transforms.CenterCrop(64),

    transforms.ToTensor(),

])



celeba_data = torchvision.datasets.ImageFolder(

    root=str(dataset_root),

    transform=transform,

)



X_img, X_class = celeba_data[0]



print("data_root = 'local_datasets/celeba'")

print('number of images:', len(celeba_data))

print('classes:', celeba_data.classes)

print('class_to_idx:', celeba_data.class_to_idx)

print('X_img shape:', tuple(X_img.shape))

print('X_class:', X_class)

print('class name:', celeba_data.classes[X_class])

print('Because there is only one folder, every label is 0.')


data_root = 'local_datasets/celeba'
number of images: 202599
classes: ['img_align_celeba']
class_to_idx: {'img_align_celeba': 0}
X_img shape: (3, 64, 64)
X_class: 0
class name: img_align_celeba
Because there is only one folder, every label is 0.
